# Spark Connect: query the lakehouse from Jupyter

This notebook is a thin **Spark Connect** client. The queries run in the cluster, on the
`spark-connect` driver and its executor (README step 12). The notebook never holds a storage
key or the Polaris credential: the server's catalog `lake` already has them.

Setup on the laptop, from the repository root. The root `pyproject.toml` pins
`pyspark-client==4.1.4` (must match the server, Spark 4.1.4) and `jupyterlab`:

```bash
uv sync
uv run jupyter lab notebooks/sample.ipynb
```

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# NodePort of the `spark-connect` Service. Inside the cluster: sc://spark-connect:15002
spark = SparkSession.builder.remote("sc://192.168.49.2:31502").getOrCreate()
spark.version

## What's in the catalog

In [ ]:
spark.sql("SHOW NAMESPACES IN lake").show()

In [ ]:
spark.sql("SHOW TABLES IN lake.silver").show(truncate=False)

In [ ]:
spark.sql("SELECT count(*) FROM lake.silver.quotes_daily").show()

## DataFrame API

The same query plan is built locally and executed by the server.

In [ ]:
daily = spark.table("lake.silver.quotes_daily")
daily.printSchema()

(daily
 .groupBy(F.to_date("created_at").alias("day"))
 .count()
 .orderBy("day")
 .show())

## Iceberg metadata tables

Each `quotes_daily` run is a commit. A rerun of the same day shows up as an `overwrite`
whose `added` and `deleted` counts are equal (README step 10).

In [ ]:
(spark.table("lake.silver.quotes_daily.snapshots")
 .select("committed_at", "snapshot_id", "operation",
         F.col("summary")["added-records"].alias("added"),
         F.col("summary")["deleted-records"].alias("deleted"),
         F.col("summary")["total-records"].alias("total"))
 .orderBy("committed_at")
 .show(truncate=False))

In [ ]:
spark.table("lake.silver.quotes_daily.partitions").show(truncate=False)

## To pandas

`toPandas()` brings the result to the notebook over Arrow, so aggregate first.

Python UDFs (`@udf`, `mapInPandas`) also work, but they run in the executor's Python
(3.10 in `apache/spark:4.1.4`), so they fail from this notebook's Python 3.13.
SQL and DataFrame code is unaffected.

In [ ]:
pdf = (daily
       .groupBy(F.to_date("created_at").alias("day"))
       .count()
       .orderBy("day")
       .toPandas())
pdf

Closes this client session only. The server keeps running for other notebooks.

In [ ]:
spark.stop()